In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
from jinja2 import Environment, FileSystemLoader

class SpriteFrameAtlas:
  atlas_path: Path
  frame_width: int
  frame_height: int
  cols: int
  rows: int
  rotations: list[int]
  animation_name: str
  def __init__(self, atlas_path: Path, frame_width: int, frame_height: int, cols: int, rows: int, rotations: list[int], animation_name: str) -> None:
    self.atlas_path = atlas_path
    self.frame_width = frame_width
    self.frame_height = frame_height
    self.cols = cols
    self.rows = rows
    self.rotations = rotations
    self.animation_name = animation_name
  def __repr__(self):
    return (f"SpriteImage(atlas_path={self.atlas_path}, frame_width={self.frame_width}, frame_height={self.frame_height}, " + 
            f"cols={self.cols}, rows={self.rows}, rotations={self.rotations}, animation_name={self.animation_name})")
project_root: Path = Path("S:/src/Richard/Godot/unknown-horizon-godot")

In [35]:
file_to_merge: str = r"S:\src\Richard\unknown-horizons\content\gfx\units\carrier\as_carrier0"
save_path: str = r"S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collectors\BuildingCollector\Sprites"

In [6]:
def merge_atlas(rows: list[list[Path]]) -> Image:
  atlas: Image = Image.new("RGBA", (0, 0))
  paste_position_y: int = 0
  for row in rows:
    paste_position_x = 0
    for file in row:
      png = Image.open(file)
      expand_width: int = max(paste_position_x + png.width - atlas.width, 0)
      expand_height: int = max(paste_position_y + png.height - atlas.height, 0)
      atlas = ImageOps.expand(atlas, (0, 0, expand_width, expand_height), (0,0,0,0))
      atlas.paste(png, (paste_position_x, paste_position_y))
      paste_position_x += png.width
    paste_position_y = atlas.height
  return atlas


In [36]:
path_to_merge_file: Path = Path(file_to_merge)
atlases: list[SpriteFrameAtlas] = []

for state_folder in path_to_merge_file.glob("*"): # got through each stated-grouped folder
  if state_folder.is_dir():
    rotations_list: list[list[Path]] = [] # the list of with lists representing different rotations
    for rotation_folder in state_folder.glob("**"): # go through the folders that represent rotations
      if rotation_folder.is_dir():
        are_all_pngs: bool = True # tells if all of the contents of the folder are pngs(to determine if it is a rotation grouping folder)
        pngs_list: list[Path] = []
        for png in rotation_folder.glob("*"): # go through the pngs in the rotation group
          if png.suffix == ".png":
            pngs_list.append(png)
          elif png.is_dir():
            are_all_pngs = False
            break

        if are_all_pngs: # if all contents are files then it is a bottom folder
          rotations_list.append(pngs_list)
    # get the merged atlas
    atlas_png: Image = merge_atlas(rotations_list)
    # get a random image for getting the width and height
    random_image_path: Path = rotations_list[0][0]
    random_image: Image = Image.open(random_image_path)
    # get the name and state of the atlas
    building_name_underscored: str = Path(save_path).parent.stem[0].lower()
    for character in Path(save_path).parent.stem[1:]:
      if character.isupper():
          building_name_underscored += '_'
      building_name_underscored += character.lower()
    tier: str = "sailors" if "units" in file_to_merge else file_to_merge.split("buildings")[-1].split("\\")[1]
    # tier: str = file_to_merge.split("buildings")[-1].split("units")[-1].split("\\")[1]
    frame_state: str = state_folder.stem.lower()
    animation_name: str = f"{tier}_{frame_state}"
    # get the path at which the atlas will be saved
    atlas_path: Path = Path(save_path) / Path(rf"{building_name_underscored}_{animation_name}_atlas.png")
    # get the width and height of a usual image
    frame_width: int = random_image.width
    frame_height: int = random_image.height
    # get the number of columns and rows
    cols: int = int(atlas_png.size[0] / frame_width)
    rows: int = int(atlas_png.size[1] / frame_height)
    # get the list of different rotations for the atlas
    rotation_amount: int = len(rotations_list)
    assert rotation_amount % 4 == 0
    rotations: list[int] = [rotation*45 for rotation in range(2 - int(rotation_amount/4), 8, 3 - int(rotation_amount/4))]
    # save the atlas
    atlas_png.save(atlas_path, "PNG")
    # create a sprite frame atlas for the atlas and add it to the list of sprite frame atlases
    atlas: SpriteFrameAtlas = SpriteFrameAtlas(atlas_path, frame_width, frame_height, cols, rows, rotations, animation_name)
    atlases.append(atlas)
print(atlases)

[SpriteImage(atlas_path=S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collectors\BuildingCollector\Sprites\building_collector_sailors_idle_atlas.png, frame_width=32, frame_height=42, cols=1, rows=8, rotations=[0, 45, 90, 135, 180, 225, 270, 315], animation_name=sailors_idle), SpriteImage(atlas_path=S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collectors\BuildingCollector\Sprites\building_collector_sailors_idle_full_atlas.png, frame_width=32, frame_height=42, cols=1, rows=8, rotations=[0, 45, 90, 135, 180, 225, 270, 315], animation_name=sailors_idle_full), SpriteImage(atlas_path=S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collectors\BuildingCollector\Sprites\building_collector_sailors_move_atlas.png, frame_width=32, frame_height=42, cols=4, rows=8, rotations=[0, 45, 90, 135, 180, 225, 270, 315], animation_name=sailors_move), SpriteImage(atlas_path=S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collecto

In [30]:
sprite_frames_template: str = """
[gd_resource type="SpriteFrames" load_steps=1{#let the godot engine correct it#} format=3 uid="uid://{{images[0].atlas_path.parent.parent.stem + "Frames"}}"]

{% for atlas in images -%}
[ext_resource type="Texture2D" uid="uid://{{atlas.animation_name}}_atlas" path="res://{{atlas.atlas_path.relative_to("S:/src/Richard/Godot/unknown-horizon-godot").as_posix()}}" id="{{atlas.animation_name}}_atlas"]
{% endfor %}
{% for atlas in images %}
  {%- for row in range(atlas.rows|int)%}
    {%- for col in range(atlas.cols|int)%}

[sub_resource type="AtlasTexture" id="{{atlas.animation_name}}_{{row}}_{{col}}"]
atlas = ExtResource("{{atlas.animation_name}}_atlas")
region = Rect2({{col * atlas.frame_width}}, {{row * atlas.frame_height}}, {{atlas.frame_width}}, {{atlas.frame_height}})
    {%- endfor %}
  {%- endfor %}
{% endfor %}
[resource]
animations = [{
"frames": [{
"duration": 0.01,
"texture": null
}],
"loop": false,
"name": &"Empty",
"speed": 5.0
},
{%- for atlas in images %}
  {%- for row in range(0, atlas.rows | int)%}
{
"frames": [
    {%- for col in range(0, atlas.cols | int)%}
{
"duration": 1.0,
"texture": SubResource("{{atlas.animation_name}}_{{row}}_{{col}}")
},
    {%- endfor %}
],
"loop": true,
"name": &"{{atlas.animation_name}}_{{atlas.rotations[row%(atlas.rotations|length)]}}",{#remember that the first charecters of the atlas name are the name of bulding which can be determined from the path#}
"speed": 5.0
},
  {%- endfor %}
{% endfor %}
]
"""

In [9]:
def write_template(output_path: Path, template_str: str, args: dict):
  print(f"Writing template: {output_path}")
  env = Environment(loader=FileSystemLoader("."))
  template = env.from_string(template_str.strip())
  rendered_content = template.render(args)
  output_path.write_text(rendered_content)

In [37]:
write_template(Path(save_path) / Path(Path(save_path).parent.stem + "Frames.tres"), sprite_frames_template, {"images": atlases})

Writing template: S:\src\Richard\Godot\unknown-horizon-godot\Assets\World\Components\Collectors\BuildingCollector\Sprites\BuildingCollectorFrames.tres
